# Prueba técnica de Ciencia de Datos — Conecta+

## Objetivo

Construir una solución reproducible para explorar los datos de Conecta+ y, en etapas posteriores, desarrollar un modelo de ciencia de datos.

En esta primera etapa exploraremos únicamente la estructura de la base SQLite: comprobaremos que el archivo exista, identificaremos sus tablas y revisaremos cuántos registros y qué columnas contiene cada una. No mostraremos datos personales ni analizaremos todavía los archivos JSONL.

> **Nota para Google Colab:** antes de ejecutar estas celdas, los archivos del proyecto deben estar disponibles en el entorno de Colab y la carpeta `datos_conecta/` debe conservarse junto a este notebook. La carga o descompresión se implementará en una etapa posterior.

In [17]:
from pathlib import Path
import sqlite3
from contextlib import closing

In [18]:
db_path = Path("datos_conecta") / "conecta.db"
print(f"Ruta esperada: {db_path}")
print(f"¿Existe el archivo?: {db_path.is_file()}")

if not db_path.is_file():
    raise FileNotFoundError(
        f"No se encontró la base SQLite en: {db_path.resolve()}"
    )

db_uri = db_path.resolve().as_uri() + "?mode=ro"

Ruta esperada: datos_conecta\conecta.db
¿Existe el archivo?: True


In [19]:
with closing(sqlite3.connect(db_uri, uri=True)) as con:
    table_names = [
        row[0]
        for row in con.execute(
            "SELECT name FROM sqlite_master "
            "WHERE type = 'table' AND name NOT LIKE 'sqlite_%' "
            "ORDER BY name"
        )
    ]

print(f"Tablas encontradas ({len(table_names)}):")
for table_name in table_names:
    print(f"- {table_name}")

Tablas encontradas (5):
- clientes
- clientes_a_puntuar
- etiquetas_entrenamiento
- facturas
- suscripciones


In [20]:
with closing(sqlite3.connect(db_uri, uri=True)) as con:
    for table_name in table_names:
        quoted_name = '"' + table_name.replace('"', '""') + '"'
        row_count = con.execute(
            f"SELECT COUNT(*) FROM {quoted_name}"
        ).fetchone()[0]
        columns = [
            row[1]
            for row in con.execute(
                f"PRAGMA table_info({quoted_name})"
            )
        ]

        print(f"\nTabla: {table_name}")
        print(f"Número de registros: {row_count}")
        print(f"Columnas ({len(columns)}): {columns}")


Tabla: clientes
Número de registros: 3000
Columnas (10): ['cliente_id', 'fecha_alta', 'segmento', 'ciudad', 'edad', 'canal_adquisicion', 'plataforma_principal', 'estado_actual', 'contactado_por_retencion', 'fecha_contacto_retencion']

Tabla: clientes_a_puntuar
Número de registros: 1549
Columnas (1): ['cliente_id']

Tabla: etiquetas_entrenamiento
Número de registros: 4758
Columnas (3): ['cliente_id', 'fecha_corte', 'churn_90d']

Tabla: facturas
Número de registros: 34267
Columnas (9): ['factura_id', 'cliente_id', 'suscripcion_id', 'periodo', 'monto', 'fecha_vencimiento', 'fecha_pago', 'metodo_pago', 'fecha_carga']

Tabla: suscripciones
Número de registros: 3362
Columnas (6): ['suscripcion_id', 'cliente_id', 'plan', 'fecha_inicio', 'fecha_fin', 'motivo_cancelacion']


# Auditoría inicial de calidad de datos

Esta sección audita las cinco tablas SQLite sin modificar registros ni mostrar filas completas. Revisaremos esquema, valores nulos, claves duplicadas, rangos, fechas y relaciones. Cuando no exista una PK declarada, distinguiremos las claves candidatas operativas de las restricciones formales del esquema.

In [ ]:
audit_findings = {}

def quoted_identifier(name):
    return '"' + name.replace('"', '""') + '"'

with closing(sqlite3.connect(db_uri, uri=True)) as con:
    audit_findings['columnas_esquema'] = 0
    for table_name in table_names:
        quoted_table = quoted_identifier(table_name)
        info = list(con.execute(f'PRAGMA table_info({quoted_table})'))
        audit_findings['columnas_esquema'] += len(info)
        print(f'\nTabla: {table_name}')
        for _, name, data_type, not_null, _, primary_key in info:
            print(f'  - {name}: tipo={data_type or "(no declarado)"}; permite_nulos={not not_null}; clave_primaria={bool(primary_key)}')

In [ ]:
with closing(sqlite3.connect(db_uri, uri=True)) as con:
    audit_findings['celdas_nulas'] = 0
    for table_name in table_names:
        quoted_table = quoted_identifier(table_name)
        print(f'\nNulos en {table_name}:')
        for _, column_name, *_ in con.execute(f'PRAGMA table_info({quoted_table})'):
            quoted_column = quoted_identifier(column_name)
            count = con.execute(f'SELECT COUNT(*) FROM {quoted_table} WHERE {quoted_column} IS NULL').fetchone()[0]
            audit_findings['celdas_nulas'] += count
            print(f'  - {column_name}: {count}')

In [ ]:
candidate_keys = {
    'clientes': [('cliente_id',)],
    'clientes_a_puntuar': [('cliente_id',)],
    'etiquetas_entrenamiento': [('cliente_id', 'fecha_corte')],
    'facturas': [('factura_id',)],
    'suscripciones': [('suscripcion_id',)],
}
with closing(sqlite3.connect(db_uri, uri=True)) as con:
    audit_findings['grupos_clave_duplicados'] = 0
    audit_findings['filas_clave_duplicadas'] = 0
    for table_name, keys in candidate_keys.items():
        quoted_table = quoted_identifier(table_name)
        for key in keys:
            grouped = ', '.join(quoted_identifier(name) for name in key)
            groups = con.execute(f'SELECT COUNT(*) FROM (SELECT {grouped} FROM {quoted_table} GROUP BY {grouped} HAVING COUNT(*) > 1)').fetchone()[0]
            rows = con.execute(f'SELECT COALESCE(SUM(n), 0) FROM (SELECT COUNT(*) AS n FROM {quoted_table} GROUP BY {grouped} HAVING COUNT(*) > 1)').fetchone()[0]
            audit_findings['grupos_clave_duplicados'] += groups
            audit_findings['filas_clave_duplicadas'] += rows
            print(f'{table_name} ({", ".join(key)}): grupos={groups}; filas={rows}')

In [ ]:
with closing(sqlite3.connect(db_uri, uri=True)) as con:
    checks = {
        'edades_fuera_de_rango': 'SELECT COUNT(*) FROM clientes WHERE edad IS NOT NULL AND (edad < 0 OR edad > 120)',
        'montos_negativos': 'SELECT COUNT(*) FROM facturas WHERE monto IS NOT NULL AND monto < 0',
        'montos_iguales_a_cero': 'SELECT COUNT(*) FROM facturas WHERE monto IS NOT NULL AND monto = 0',
        'contactos_antes_del_alta': 'SELECT COUNT(*) FROM clientes WHERE fecha_contacto_retencion IS NOT NULL AND fecha_alta IS NOT NULL AND date(fecha_contacto_retencion) < date(fecha_alta)',
        'finalizacion_antes_del_inicio': 'SELECT COUNT(*) FROM suscripciones WHERE fecha_fin IS NOT NULL AND fecha_inicio IS NOT NULL AND date(fecha_fin) < date(fecha_inicio)',
    }
    for label, query in checks.items():
        audit_findings[label] = con.execute(query).fetchone()[0]
        print(f'{label}: {audit_findings[label]}')
    date_columns = {'clientes': ['fecha_alta', 'fecha_contacto_retencion'], 'etiquetas_entrenamiento': ['fecha_corte'], 'facturas': ['fecha_vencimiento', 'fecha_pago', 'fecha_carga'], 'suscripciones': ['fecha_inicio', 'fecha_fin']}
    audit_findings['fechas_no_interpretables'] = 0
    for table_name, columns in date_columns.items():
        for column_name in columns:
            qtable = quoted_identifier(table_name); qcolumn = quoted_identifier(column_name)
            count = con.execute(f'SELECT COUNT(*) FROM {qtable} WHERE {qcolumn} IS NOT NULL AND date({qcolumn}) IS NULL').fetchone()[0]
            audit_findings['fechas_no_interpretables'] += count
            print(f'{table_name}.{column_name}: fechas_no_interpretables={count}')
    minimum, maximum = con.execute('SELECT MIN(monto), MAX(monto) FROM facturas WHERE monto IS NOT NULL').fetchone()
    print(f'facturas.monto: minimo={minimum}; maximo={maximum}')

In [ ]:
relationship_checks = {
    'suscripciones_sin_cliente': 'SELECT COUNT(*) FROM suscripciones s LEFT JOIN clientes c ON c.cliente_id = s.cliente_id WHERE c.cliente_id IS NULL',
    'facturas_sin_cliente': 'SELECT COUNT(*) FROM facturas f LEFT JOIN clientes c ON c.cliente_id = f.cliente_id WHERE c.cliente_id IS NULL',
    'facturas_sin_suscripcion': 'SELECT COUNT(*) FROM facturas f LEFT JOIN suscripciones s ON s.suscripcion_id = f.suscripcion_id WHERE s.suscripcion_id IS NULL',
    'facturas_cliente_distinto_de_suscripcion': 'SELECT COUNT(*) FROM facturas f JOIN suscripciones s ON s.suscripcion_id = f.suscripcion_id WHERE f.cliente_id <> s.cliente_id',
    'clientes_a_puntuar_sin_cliente': 'SELECT COUNT(*) FROM clientes_a_puntuar p LEFT JOIN clientes c ON c.cliente_id = p.cliente_id WHERE c.cliente_id IS NULL',
    'etiquetas_sin_cliente': 'SELECT COUNT(*) FROM etiquetas_entrenamiento e LEFT JOIN clientes c ON c.cliente_id = e.cliente_id WHERE c.cliente_id IS NULL',
}
with closing(sqlite3.connect(db_uri, uri=True)) as con:
    for label, query in relationship_checks.items():
        audit_findings[label] = con.execute(query).fetchone()[0]
        print(f'{label}: {audit_findings[label]}')

## Resumen de la auditoría SQLite

Los valores distintos de cero son señales para la etapa de limpieza; esta auditoría no corrige registros.

In [ ]:
print('Resumen cuantitativo SQLite:')
for label, count in audit_findings.items():
    print(f'- {label}: {count}')

# Exploración y normalización de datos no estructurados

Esta sección trabaja sobre copias en memoria de `tickets.jsonl` y `eventos.jsonl.gz`. No se muestran textos personales ni se modifican los archivos fuente. En Colab, usa `%pip install -q pandas matplotlib` si faltan dependencias.

Las fechas se normalizan a UTC cuando es posible. Las fechas sin zona horaria se documentan como un supuesto: no se inventa una zona; se conservan como `sin_zona` y requieren una decisión de negocio. Para variables históricas se filtrará toda observación posterior a la fecha de corte.

In [ ]:
import gzip
import hashlib
import json
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

tickets_path = db_path.parent / 'tickets.jsonl'
events_path = db_path.parent / 'eventos.jsonl.gz'

def load_jsonl(path, compressed=False):
    opener = gzip.open if compressed else open
    with opener(path, 'rt', encoding='utf-8') as file:
        return [json.loads(line) for line in file if line.strip()]

def record_fingerprint(record):
    text = json.dumps(record, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

tickets_raw = load_jsonl(tickets_path)
events_raw = load_jsonl(events_path, compressed=True)
print(f'tickets: {len(tickets_raw):,}')
print(f'eventos: {len(events_raw):,}')

In [ ]:
def field_profile(records):
    rows = []
    total = len(records)
    for field in sorted({key for record in records for key in record}):
        present = [record[field] for record in records if field in record]
        non_null = [value for value in present if value is not None]
        rows.append({'campo': field, 'presentes': len(present), 'nulos': sum(value is None for value in present), 'ausentes': total - len(present), 'tipos': ', '.join(sorted({type(value).__name__ for value in non_null}))})
    return pd.DataFrame(rows)

print(field_profile(tickets_raw).to_string(index=False))
print('Variantes:', dict(Counter('moderno' if 'ticket_id' in r else 'legado' for r in tickets_raw)))
print('Esquemas distintos:', len({tuple(sorted(r)) for r in tickets_raw}))
ticket_hashes = pd.Series([record_fingerprint(r) for r in tickets_raw])
print('Tickets duplicados exactos adicionales:', int(ticket_hashes.duplicated().sum()))
for field in ['ticket_id', 'id']:
    values = [r.get(field) for r in tickets_raw if r.get(field) is not None]
    print(f'{field}: repetidos={len(values) - len(set(values))}; ausentes={len(tickets_raw) - len(values)}')

In [ ]:
def nested(record, field, key):
    value = record.get(field)
    return value.get(key) if isinstance(value, dict) else None

def customer_text(record):
    if isinstance(record.get('customer'), str):
        return record['customer']
    messages = record.get('mensajes', [])
    return ' '.join(item.get('texto', '') for item in messages if isinstance(item, dict) and isinstance(item.get('texto'), str)) or None

ticket_rows = []
for record in tickets_raw:
    ticket_rows.append({'ticket_id': record.get('ticket_id', record.get('id')), 'cliente_id': nested(record, 'cliente', 'id'), 'created_at_raw': record.get('created_at', record.get('fecha')), 'resolved_at_raw': record.get('resolved_at'), 'category': record.get('category', nested(record, 'tipo', 'principal')), 'channel': record.get('channel', record.get('canal')), 'status': record.get('status', nested(record, 'estado', 'valor')), 'satisfaction': record.get('csat', nested(record, 'satisfaccion', 'csat')), 'customer_text': customer_text(record), 'schema_variant': 'moderno' if 'ticket_id' in record else 'legado'})
tickets_df = pd.DataFrame(ticket_rows)
tickets_df['created_at_utc'] = pd.to_datetime(tickets_df['created_at_raw'], errors='coerce', utc=True, format='mixed')
tickets_df['resolved_at_utc'] = pd.to_datetime(tickets_df['resolved_at_raw'], errors='coerce', utc=True, format='mixed')
tickets_df['created_timezone'] = tickets_df['created_at_raw'].astype('string').str.extract(r'(Z|[+-]\d{2}:?\d{2})$', expand=False).fillna('sin_zona')
print('tickets_df:', tickets_df.shape)
print('Nulos por columna:')
print(tickets_df.isna().sum().to_string())
print('Texto presente:', int(tickets_df['customer_text'].notna().sum()), 'de', len(tickets_df))
print('Zonas de fecha de creación:')
print(tickets_df['created_timezone'].value_counts().to_string())
print('Categorías:')
print(tickets_df['category'].value_counts().to_string())
print('Estados:')
print(tickets_df['status'].value_counts().to_string())

In [ ]:
events_df = pd.DataFrame(events_raw).copy()
events_df['ts_raw'] = events_df['ts']
events_df['ts_utc'] = pd.to_datetime(events_df['ts'], errors='coerce', utc=True, format='mixed')
events_df['timezone_token'] = events_df['ts_raw'].str.extract(r'(Z|[+-]\d{2}:?\d{2})$', expand=False).fillna('sin_zona')
event_hashes = pd.Series([record_fingerprint(r) for r in events_raw])
print('events_df:', events_df.shape)
print('Nulos por columna:')
print(events_df.isna().sum().to_string())
print('Duplicados exactos adicionales:', int(event_hashes.duplicated().sum()))
print('Fechas no interpretables:', int(events_df['ts_utc'].isna().sum()))
print('Zonas:', events_df['timezone_token'].value_counts().to_string())
print('Tipos:', events_df['tipo'].value_counts().to_string())
print('Plataformas:', events_df['plataforma'].value_counts().to_string())
print('Errores:', events_df['codigo_error'].fillna('sin_error').value_counts().to_string())

## Relación con clientes y prevención de fuga temporal

Los identificadores se comparan con SQLite. El conteo posterior a la fecha de corte se interpreta como pares evento-corte, porque un cliente puede tener varias fechas de corte.

In [ ]:
with closing(sqlite3.connect(db_uri, uri=True)) as con:
    clients = pd.read_sql_query('SELECT cliente_id FROM clientes', con)
    cutoffs = pd.read_sql_query('SELECT cliente_id, fecha_corte FROM etiquetas_entrenamiento', con)
known_ids = set(clients['cliente_id'].astype('int64'))
event_ids = set(events_df['cliente_id'].astype('int64'))
ticket_ids = set(tickets_df['cliente_id'].dropna().astype('int64'))
print('Clientes de eventos ausentes en SQLite:', len(event_ids - known_ids))
print('Clientes de tickets ausentes en SQLite:', len(ticket_ids - known_ids))
cutoffs['fecha_corte_utc'] = pd.to_datetime(cutoffs['fecha_corte'], errors='coerce', utc=True, format='mixed')
pairs = events_df[['cliente_id', 'ts_utc']].merge(cutoffs[['cliente_id', 'fecha_corte_utc']], on='cliente_id', how='inner')
future_pairs = pairs['ts_utc'] > pairs['fecha_corte_utc']
print('Pares evento-corte evaluados:', len(pairs))
historical_event_pairs = pairs.loc[~future_pairs].copy()
assert not (historical_event_pairs['ts_utc'] > historical_event_pairs['fecha_corte_utc']).fillna(False).any()
print('Pares posteriores al corte:', int(future_pairs.sum()))
print('Pares conservados para históricos:', len(historical_event_pairs))
print('Cortes no interpretables:', int(cutoffs['fecha_corte_utc'].isna().sum()))

In [ ]:
ticket_summary = tickets_df.groupby(['schema_variant', 'status'], dropna=False).size().reset_index(name='tickets')
event_summary = events_df.groupby(['tipo', 'plataforma'], dropna=False).size().reset_index(name='eventos')
print(ticket_summary.to_string(index=False))
print(event_summary.to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
tickets_df['status'].value_counts().plot.bar(ax=axes[0], title='Tickets por estado')
events_df['tipo'].value_counts().plot.bar(ax=axes[1], title='Eventos por tipo')
plt.tight_layout()
plt.show()

## Decisiones y pendientes

No se eliminan duplicados. El texto libre se conserva en memoria pero no se muestra. Las fechas sin zona quedan documentadas como `sin_zona`; no se les asigna una zona arbitraria. Antes de construir variables históricas se debe filtrar por `cliente_id` y `fecha_corte`, excluyendo pares posteriores al corte. El entrenamiento del modelo queda fuera de esta etapa.